# DORA-Informed ICT Incident Monitor: data analysis

This notebook explains the historical operations dataset used by the dashboard. It does not classify historical tickets under DORA. The financial-sector examples are fictional and stored separately.

**Source:** Amaral, Fantinato & Peres (2018), [UCI Incident management process enriched event log](https://doi.org/10.24432/C57S4H), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). The anonymised records came from an IT company's ServiceNow system.

Run from the repository root with `pip install -r requirements-dev.txt`, then execute cells in order. The next cell downloads the original UCI CSV if needed and verifies its SHA-256. For offline use, set `DORA_SOURCE_CSV` to an existing copy. `prepare_data.py` is the only processing implementation; this notebook neither writes application code nor copies its transformation logic.


In [1]:
from pathlib import Path
import hashlib
import io
import os
import sys
import urllib.request
import zipfile
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'prepare_data.py').is_file():
    raise RuntimeError('Start Jupyter from the repository root containing prepare_data.py.')
sys.path.insert(0, str(ROOT))
from prepare_data import prepare, boolean_series

SOURCE_URL = 'https://archive.ics.uci.edu/static/public/498/incident+management+process+enriched+event+log.zip'
EXPECTED_HASH = 'fd184bbfd62329cfe093e99da2ea7071905f2ead91900b448eb2635870821bef'
source_path = Path(os.environ.get('DORA_SOURCE_CSV', ROOT / 'data/source/incident_event_log.csv'))
if not source_path.exists():
    if os.environ.get('DORA_SOURCE_CSV'):
        raise FileNotFoundError(source_path)
    with urllib.request.urlopen(SOURCE_URL, timeout=60) as response:
        archive = zipfile.ZipFile(io.BytesIO(response.read()))
    source_bytes = archive.read('incident_event_log.csv')
    if hashlib.sha256(source_bytes).hexdigest() != EXPECTED_HASH:
        raise ValueError('Downloaded source hash differs from the validated UCI file.')
    source_path.parent.mkdir(parents=True, exist_ok=True)
    source_path.write_bytes(source_bytes)
source_hash = hashlib.sha256(source_path.read_bytes()).hexdigest()
assert source_hash == EXPECTED_HASH, 'Unexpected source file: review provenance before continuing.'
raw = pd.read_csv(source_path, low_memory=False, keep_default_na=False)
print(f'Source SHA-256: {source_hash}')
print(f'Events: {len(raw):,}; columns: {len(raw.columns)}; incident IDs: {raw.number.nunique():,}')


Source SHA-256: fd184bbfd62329cfe093e99da2ea7071905f2ead91900b448eb2635870821bef
Events: 141,712; columns: 36; incident IDs: 24,918


## Structure and exploratory findings

A row is a ticket snapshot, not an independent incident. Repeated incident IDs represent lifecycle updates. The table below summarises update frequency and the distribution of latest priorities after preparation. An exact duplicate is different from a repeated ID or two updates sharing a timestamp.


In [2]:
display(raw.groupby('number').size().describe().rename('Events per incident').to_frame())
print(f'Exact duplicate event rows: {raw.duplicated().sum():,}')
print(f'Updates sharing incident ID and update timestamp: {raw.duplicated(["number", "sys_updated_at"]).sum():,}')


,Events per incident
count,24918.000000
mean,5.687134
std,3.677845
min,2.000000
25%,3.000000
50%,5.000000
75%,7.000000
max,58.000000


Exact duplicate event rows: 0
Updates sharing incident ID and update timestamp: 14,118


## Missing values, timestamps and incident preparation

`prepare()` converts question marks and whitespace-only values to missing data, parses operational dates using day/month/year, and validates incident identifiers and numeric revision counts. It selects one complete latest snapshot per incident, ordered by revision count, update timestamp and stable source order. It does not combine values from different snapshots.

Resolution time uses opened-to-resolved timestamps only for resolved/closed snapshots with a non-negative interval. Missing resolution dates are not replaced with closure dates. SLA values use nullable booleans, so unknown does not become False. Date parse failures and negative intervals are counted separately from missing dates.


In [3]:
incidents, quality = prepare(raw)
missing = pd.Series(quality['missing_by_column'], name='Missing events').sort_values(ascending=False)
display(missing[missing.gt(0)].to_frame().assign(Percent=lambda d: (d['Missing events'] / len(raw) * 100).round(2)))
display(pd.Series(quality['invalid_datetimes'], name='Invalid non-missing timestamps').to_frame())
display(incidents.priority.value_counts().sort_index().rename('Incidents').to_frame())


,Missing events,Percent
caused_by,141689,99.98
vendor,141468,99.83
cmdb_ci,141267,99.69
rfc,140721,99.30
problem_id,139417,98.38
sys_created_at,53076,37.45
sys_created_by,53076,37.45
u_symptom,32964,23.26
assigned_to,27496,19.40
assignment_group,14213,10.03


,Invalid non-missing timestamps
opened_at,0
sys_updated_at,0
resolved_at,0
closed_at,0


,Incidents
priority,
1 - Critical,270
2 - High,408
3 - Moderate,23466
4 - Low,774


## Data-integrity validation

These checks compare the freshly prepared data with the dataset committed for deployment. They fail if the source dimensions, unique IDs, known missing resolution count or generated dataset differ. The operational and regulatory tests remain in `tests/`; run `python -m pytest -q` separately.


In [4]:
assert raw.shape == (141712, 36)
assert len(incidents) == incidents.number.nunique() == 24918
assert incidents.number.notna().all()
assert quality['missing_resolution_duration'] == 1556
assert quality['negative_durations'] == 0
assert incidents.resolution_duration_hours.dropna().ge(0).all()
committed = pd.read_csv(ROOT / 'data/incidents.csv', low_memory=False)
# Compare equivalent CSV round-trips, including column order and missing values.
fresh = pd.read_csv(io.StringIO(incidents.to_csv(index=False)), low_memory=False)
pd.testing.assert_frame_equal(fresh, committed)
print('Validated: 141,712 events → 24,918 unique incidents; generated data matches deployment data.')


Validated: 141,712 events → 24,918 unique incidents; generated data matches deployment data.


## Operational metrics

Median describes the typical recorded resolution interval; the mean is sensitive to long-running tickets. Both exclude missing resolution durations. SLA rate divides known True outcomes by all known outcomes. These are historical ticket metrics, not evidence of regulatory compliance. All latest snapshots in this sample are Closed, so closed count adds no information beyond total incidents.


In [5]:
resolution = incidents.resolution_duration_hours.dropna()
sla = boolean_series(incidents.made_sla).dropna()
metrics = pd.Series({
    'Unique incidents': len(incidents),
    'Valid resolution intervals': len(resolution),
    'Missing resolution intervals': int(incidents.resolution_duration_hours.isna().sum()),
    'Median resolution (hours)': resolution.median(),
    'Mean resolution (hours)': resolution.mean(),
    'Resolution data coverage (%)': len(resolution) / len(incidents) * 100,
    'Known SLA outcomes': len(sla),
    'SLA met': int(sla.sum()),
    'SLA not met': int((~sla).sum()),
    'SLA met rate (%)': sla.mean() * 100,
})
display(metrics.rename('Value').to_frame().round(3))
display(incidents.incident_state.value_counts().rename('Latest snapshots').to_frame())


,Value
Unique incidents,24918.000
Valid resolution intervals,23362.000
Missing resolution intervals,1556.000
Median resolution (hours),22.100
Mean resolution (hours),178.172
Resolution data coverage (%),93.756
Known SLA outcomes,24918.000
SLA met,15803.000
SLA not met,9115.000
SLA met rate (%),63.420


,Latest snapshots
incident_state,
Closed,24918


## DORA data limitations

The source does not establish financial-service criticality, affected-client denominators, transaction impact, EU Member State impact, malicious-access evidence or regulatory reporting timestamps. Ticket resolution time is neither DORA incident duration nor service downtime. Historical SLA outcomes are not DORA classifications.

The dashboard's five fictional scenarios use `rules.py` and `data/dora_scenarios.json`. Unknown facts remain unknown. Regulatory interpretation, recurring-incident review, entity applicability and holiday exceptions require responsible human review. See the app's Reference Guide and README for official EU sources and scope limits.
